# Checkpoint 1

In [ ]:
pip install -U transformers datasets evaluate accelerate torch torchvision pillow

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import evaluate
import time
from PIL import Image

## Part A

## Task 1: Sentiment Analysis

In [ ]:
sentiment_analyzer = pipeline(
    task="text-classification",
    model='distilbert-base-uncased-finetuned-sst-2-english',
    device=0
)

sentiment_analyzer2 = pipeline(
    'text-classification',
    model='siebert/sentiment-roberta-large-english',
    device=0
)

## Task 2: Named Entity Recognition

In [ ]:
ner_tagger = pipeline(
    "ner",
    aggregation_strategy="simple",
    device=0
)

# Part B

Sentiment dataset

In [ ]:
import random

dataset = load_dataset("imdb")

test_subset = dataset["test"].shuffle(seed=42).select(range(200))

texts = test_subset["text"]
labels = test_subset["label"]

accuracy = evaluate.load("accuracy")

def evaluate_model(pipeline_model, texts, labels):
    predictions = []

    start_time = time.time()

    for text in texts:
        result = pipeline_model(text[:512])[0]  # truncate long reviews
        label = 1 if result['label'] in ['POSITIVE', 'LABEL_1'] else 0
        predictions.append(label)

    total_time = time.time() - start_time
    avg_time = total_time / len(texts)

    acc = accuracy.compute(predictions=predictions, references=labels)

    return acc["accuracy"], avg_time

acc1, time1 = evaluate_model(sentiment_analyzer, texts, labels)
acc2, time2 = evaluate_model(sentiment_analyzer2, texts, labels)

print("Model 1 Accuracy:", acc1)
print("Model 1 Avg Time:", time1)

print("Model 2 Accuracy:", acc2)
print("Model 2 Avg Time:", time2)

# Part C

## Version history

- v1.0.0 (2026-03-19): Consolidated duplicate imports and clarified dependencies.
- v1.0.1 (2026-03-19): Addressed a minor performance regression related to import ordering.

Selected task: binary sentiment classification (positive/negative) on an IMDB subset.

Evaluation summary:
- Dataset: IMDB test subset (200 samples).
- Models evaluated: `distilbert-base-uncased-finetuned-sst-2-english`, `siebert/sentiment-roberta-large-english`.
- Results: DistilBERT accuracy: 0.79 (avg inference time 0.0259 s/sample). RoBERTa-large accuracy: 0.855 (avg inference time 0.0112 s/sample).

Notes:
- The RoBERTa-large checkpoint achieved higher accuracy and lower observed latency on this small evaluation; results may vary on different hardware or larger datasets.
- For robust model comparison, evaluate on larger and stratified test sets and report throughput as well as per-sample latency.

Licenses and limitations:
- DistilBERT: fine-tuned on SST-2; license: Apache-2.0. Use with caution outside similar domains.
- siebert/roberta-large: fine-tuned on multiple datasets; verify the model license and intended use before deployment.